# Binary classification of `PW_0.5h`

Predict whether a failure occurs within the next 30 minutes from lagged sensor readings.

1. Read `PW_0.5h/train.csv` and `PW_0.5h/val.csv`.
2. Expand each series with `lag_window`. Each row keeps the last `READING_WINDOW` minutes of the sensors. `PW_0.5h` stays the label of the newest minute and is not used as a feature.
3. Hold out the last 20% of the lagged training rows as an eval set. Drop the 30 minutes just before that cut, because a training label looks 30 minutes ahead and would otherwise overlap the eval period.
4. Tune the tree hyperparameters with Optuna's Bayesian sampler and expanding-window time-series cross-validation at a low learning rate. Choose the learning rate and tree count on the eval set. Refit the final model on the fit rows.
5. Choose the F1 threshold on the eval holdout, then report F1 and AUC on `train.csv` and `val.csv`.

In [ ]:
import time
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier

from pipeline_functions import lag_window

DATA_DIR = Path("PW_0.5h")
TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH = DATA_DIR / "val.csv"
TARGET = "PW_0.5h"

READING_WINDOW = 30
EVAL_FRACTION = 0.20
PURGE = 30
N_SPLITS = 5
N_TRIALS = 40
RANDOM_SEED = 42

STAGE1_LEARNING_RATE = 0.1
STAGE1_N_ESTIMATORS = 500
STAGE2_N_ESTIMATORS = 2000
EARLY_STOPPING_ROUNDS = 40
LEARNING_RATES = (0.01, 0.02, 0.05, 0.1, 0.2)

/Users/kingsleychan/Projects/IBM Z Datathon/IBM_Z_Datathon/kingsleyenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load and lag

`READING_WINDOW` is 30 minutes, the same length as the failure horizon. Each file is lagged on its own, in timestamp order.

In [2]:
def load_series(path):
    frame = pd.read_csv(path, parse_dates=["timestamp"])
    return frame.set_index("timestamp").sort_index()


def summarize(name, frame):
    labels = frame[TARGET].astype(int)
    print(
        f"{name}: {len(frame):,} rows, {frame.shape[1] - 1} features, "
        f"{int(labels.sum()):,} positives ({labels.mean():.2%}), "
        f"{frame.index.min()} to {frame.index.max()}"
    )


train_raw = load_series(TRAIN_PATH)
val_raw = load_series(VAL_PATH)
train = lag_window(train_raw, READING_WINDOW)
val = lag_window(val_raw, READING_WINDOW)

summarize("train.csv", train)
summarize("val.csv", val)
lag0_columns = [column for column in train.columns if column.endswith("_lag_0") or column == TARGET]
train.loc[:, lag0_columns].head(3)

train.csv: 40,325 rows, 120 features, 810 positives (2.01%), 2023-08-01 03:34:00 to 2023-08-29 03:38:00
val.csv: 34,315 rows, 120 features, 540 positives (1.57%), 2023-08-29 04:08:00 to 2023-09-22 00:02:00


,CMS air pressure_lag_0,Concentrazione ossigeno base_lag_0,Nitrogen pressure_lag_0,Oxygen over threshold_lag_0,PW_0.5h
timestamp,,,,,
2023-08-01 03:34:00,-0.578901,-0.986817,-1.394478,-0.044569,0
2023-08-01 03:35:00,-0.578901,-0.986817,-1.394478,-0.044569,0
2023-08-01 03:36:00,1.270593,-0.986817,-1.394478,-0.044569,0


## Eval holdout

The eval set is the last 20% of lagged `train.csv`. It is used to pick the learning rate, the number of trees, and the F1 threshold. Time-series cross-validation uses only the earlier fit rows. `val.csv` stays out of fitting and tuning.

In [3]:
def features_and_labels(frame):
    labels = frame[TARGET].astype(int)
    features = frame.drop(columns=[TARGET])
    return features, labels


X_train, y_train = features_and_labels(train)
eval_start = int(len(train) * (1 - EVAL_FRACTION))
fit_end = eval_start - PURGE
if fit_end <= PURGE:
    raise ValueError("Training series is too short for the eval holdout and purge gap")

X_fit, y_fit = X_train.iloc[:fit_end], y_train.iloc[:fit_end]
X_eval, y_eval = X_train.iloc[eval_start:], y_train.iloc[eval_start:]
feature_columns = list(X_fit.columns)

SCALE_POS_WEIGHT = float((y_fit == 0).sum() / (y_fit == 1).sum())
print(
    f"fit rows {len(X_fit):,} ({int(y_fit.sum())} positives), "
    f"purged gap {PURGE}, "
    f"eval rows {len(X_eval):,} ({int(y_eval.sum())} positives)"
)
print(f"eval starts at {X_eval.index[0]}")
print(f"scale_pos_weight {SCALE_POS_WEIGHT:.2f}")

fit rows 32,230 (600 positives), purged gap 30, eval rows 8,065 (210 positives)
eval starts at 2023-08-23 13:14:00
scale_pos_weight 52.72


## Stage 1: tree hyperparameters at a low learning rate

The learning rate stays at 0.05. Optuna's TPE sampler searches depth, child weight, subsample, column sample, gamma, and L2 jointly. Each trial is scored by expanding-window time-series cross-validation. Folds come from `TimeSeriesSplit`, and each fold drops the last 30 training rows so those labels do not reach into the fold's validation block. Every candidate may stop early. The score is the mean validation AUC.

In [4]:
def make_classifier(tree_params, learning_rate, n_estimators, early_stopping_rounds=None):
    settings = dict(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        scale_pos_weight=SCALE_POS_WEIGHT,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=0,
        **tree_params,
    )
    if early_stopping_rounds is not None:
        settings["early_stopping_rounds"] = early_stopping_rounds
    return XGBClassifier(**settings)


def fold_indices(n_rows):
    splitter = TimeSeriesSplit(n_splits=N_SPLITS)
    for train_idx, valid_idx in splitter.split(np.arange(n_rows)):
        train_idx = train_idx[train_idx < valid_idx[0] - PURGE]
        if len(train_idx) == 0 or y_fit.iloc[valid_idx].nunique() < 2:
            raise ValueError("A time-series fold has no training rows or one validation class")
        yield train_idx, valid_idx


def mean_cv_auc(tree_params):
    scores = []
    trees = []
    for train_idx, valid_idx in fold_indices(len(X_fit)):
        model = make_classifier(
            tree_params,
            learning_rate=STAGE1_LEARNING_RATE,
            n_estimators=STAGE1_N_ESTIMATORS,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        )
        model.fit(
            X_fit.iloc[train_idx],
            y_fit.iloc[train_idx],
            eval_set=[(X_fit.iloc[valid_idx], y_fit.iloc[valid_idx])],
            verbose=False,
        )
        proba = model.predict_proba(X_fit.iloc[valid_idx])[:, 1]
        scores.append(roc_auc_score(y_fit.iloc[valid_idx], proba))
        trees.append(model.best_iteration + 1)
    return float(np.mean(scores)), float(np.mean(trees))


def objective(trial):
    candidate = {
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 20.0, log=True),
    }
    score, trees = mean_cv_auc(candidate)
    trial.set_user_attr("mean_best_trees", trees)
    print(f"trial {trial.number:02d}  cv_auc={score:.4f}  trees={trees:.0f}  {candidate}")
    return score


print("Time-series folds after the purge gap:")
for fold, (train_idx, valid_idx) in enumerate(fold_indices(len(X_fit)), start=1):
    print(
        f"  fold {fold}: train {len(train_idx):,} ({int(y_fit.iloc[train_idx].sum())} positives), "
        f"valid {len(valid_idx):,} ({int(y_fit.iloc[valid_idx].sum())} positives)"
    )

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED, multivariate=True),
)
started = time.perf_counter()
study.optimize(objective, n_trials=N_TRIALS)
tree_params = study.best_params
print(f"\nstage 1 finished in {time.perf_counter() - started:.0f}s")
print(f"best cv_auc={study.best_value:.4f}")
print("selected tree parameters:")
print(tree_params)

trial_results = (
    pd.DataFrame(
        [
            {
                **trial.params,
                "cv_auc": trial.value,
                "mean_best_trees": trial.user_attrs["mean_best_trees"],
            }
            for trial in study.trials
            if trial.value is not None
        ]
    )
    .sort_values("cv_auc", ascending=False)
    .reset_index(drop=True)
)
trial_results

Time-series folds after the purge gap:
  fold 1: train 5,345 (120 positives), valid 5,371 (60 positives)
  fold 2: train 10,716 (180 positives), valid 5,371 (120 positives)
  fold 3: train 16,087 (300 positives), valid 5,371 (60 positives)
  fold 4: train 21,458 (360 positives), valid 5,371 (120 positives)
  fold 5: train 26,829 (480 positives), valid 5,371 (120 positives)


/Users/kingsleychan/Projects/IBM Z Datathon/IBM_Z_Datathon/kingsleyenv/lib/python3.13/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(


trial 00  cv_auc=0.9371  trees=23  {'max_depth': 5, 'min_child_weight': 20, 'subsample': 0.9195981825434215, 'colsample_bytree': 0.8795975452591109, 'gamma': 0.15601864044243652, 'reg_lambda': 1.5957084694148351}


trial 01  cv_auc=0.9472  trees=44  {'max_depth': 3, 'min_child_weight': 18, 'subsample': 0.8803345035229626, 'colsample_bytree': 0.9124217733388136, 'gamma': 0.020584494295802447, 'reg_lambda': 18.276027831785726}


trial 02  cv_auc=0.9488  trees=47  {'max_depth': 7, 'min_child_weight': 5, 'subsample': 0.7545474901621302, 'colsample_bytree': 0.7550213529560301, 'gamma': 0.3042422429595377, 'reg_lambda': 4.816414530907083}


...

trial 36  cv_auc=0.9393  trees=67  {'max_depth': 3, 'min_child_weight': 10, 'subsample': 0.8819559201805666, 'colsample_bytree': 0.8597059640130621, 'gamma': 0.2273262801695467, 'reg_lambda': 1.20303028750163}


trial 37  cv_auc=0.9459  trees=68  {'max_depth': 3, 'min_child_weight': 11, 'subsample': 0.768993408832004, 'colsample_bytree': 0.9174074674949215, 'gamma': 0.5848206552838511, 'reg_lambda': 4.61519903876646}


trial 38  cv_auc=0.9445  trees=65  {'max_depth': 3, 'min_child_weight': 10, 'subsample': 0.9288212219135481, 'colsample_bytree': 0.8543355848203601, 'gamma': 0.6775397125915406, 'reg_lambda': 13.931214205638742}


trial 39  cv_auc=0.9509  trees=43  {'max_depth': 4, 'min_child_weight': 10, 'subsample': 0.8764904048180552, 'colsample_bytree': 0.9736136285311253, 'gamma': 0.6935034110717776, 'reg_lambda': 17.487157249274713}

stage 1 finished in 96s
best cv_auc=0.9540
selected tree parameters:
{'max_depth': 3, 'min_child_weight': 9, 'subsample': 0.8104798358669292, 'colsample_bytree': 0.9174568584378915, 'gamma': 0.17491310680137717, 'reg_lambda': 4.033873215252918}


,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_lambda,cv_auc,mean_best_trees
0,3,9,0.810480,0.917457,0.174913,4.033873,0.954029,62.6
1,3,5,0.820249,0.917434,0.489173,13.096782,0.953298,41.0
2,3,7,0.808282,0.871010,0.400129,11.214729,0.953208,58.4
3,4,9,0.823843,0.962065,0.553035,4.863373,0.952622,44.6
4,3,4,0.790311,0.902299,0.726011,17.755452,0.952502,62.8
5,4,5,0.813753,0.952233,0.034444,5.192725,0.952199,61.2
6,4,9,0.785088,0.953744,0.129017,4.527914,0.951827,65.2
7,4,9,0.829311,0.921568,0.385213,4.931741,0.951810,58.8
8,3,5,0.895552,0.807152,0.194760,16.928117,0.951383,54.2
9,4,10,0.767438,0.981813,0.317517,19.138665,0.951178,66.0


## Stage 2: learning rate on the eval set

Tree parameters stay fixed. Each learning rate is allowed up to 2000 trees and stops early on eval AUC. The learning rate with the highest eval AUC is kept, along with the tree count at that best round. A tie keeps the lower learning rate.

In [5]:
learning_rows = []
best_learning_rate = None
best_trees = None
best_eval_auc = -np.inf
started = time.perf_counter()
for learning_rate in LEARNING_RATES:
    model = make_classifier(
        tree_params,
        learning_rate=learning_rate,
        n_estimators=STAGE2_N_ESTIMATORS,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )
    model.fit(X_fit, y_fit, eval_set=[(X_eval, y_eval)], verbose=False)
    proba = model.predict_proba(X_eval)[:, 1]
    eval_auc = float(roc_auc_score(y_eval, proba))
    trees = int(model.best_iteration + 1)
    learning_rows.append(
        {"learning_rate": learning_rate, "eval_auc": eval_auc, "best_trees": trees}
    )
    print(f"learning_rate={learning_rate:<5}  eval_auc={eval_auc:.4f}  trees={trees}")
    if eval_auc > best_eval_auc:
        best_eval_auc = eval_auc
        best_learning_rate = learning_rate
        best_trees = trees

learning_results = pd.DataFrame(learning_rows).sort_values(
    ["eval_auc", "learning_rate"], ascending=[False, True]
).reset_index(drop=True)
print(f"stage 2 finished in {time.perf_counter() - started:.0f}s")
print(f"selected learning_rate={best_learning_rate}  n_estimators={best_trees}")
learning_results

learning_rate=0.01   eval_auc=0.9013  trees=235


learning_rate=0.02   eval_auc=0.9012  trees=117


learning_rate=0.05   eval_auc=0.8950  trees=71


learning_rate=0.1    eval_auc=0.8959  trees=35


learning_rate=0.2    eval_auc=0.8911  trees=11
stage 2 finished in 4s
selected learning_rate=0.01  n_estimators=235


,learning_rate,eval_auc,best_trees
0,0.01,0.901302,235
1,0.02,0.901175,117
2,0.10,0.895860,35
3,0.05,0.895000,71
4,0.20,0.891072,11


## Stage 3: final model

Refit on the fit rows with the stage-1 tree parameters, the stage-2 learning rate, and the stage-2 tree count. The eval holdout is not part of this fit.

In [6]:
final_model = make_classifier(
    tree_params,
    learning_rate=best_learning_rate,
    n_estimators=best_trees,
)
final_model.fit(X_fit, y_fit, verbose=False)
print(
    f"fitted {final_model.n_estimators} trees at learning_rate={best_learning_rate} "
    f"on {len(X_fit):,} rows"
)

fitted 235 trees at learning_rate=0.01 on 32,230 rows


## Scores on `train.csv` and `val.csv`

The positive-class threshold is the value on a 0.01 grid that maximizes F1 on the eval holdout. AUC does not use that threshold. `train.csv` includes the fit rows, so its score is partly in-sample. The eval-holdout row is the selection set. `val.csv` was not used to fit the model, choose hyperparameters, or choose the threshold.

In [7]:
def predict_proba(features):
    return final_model.predict_proba(features.loc[:, feature_columns])[:, 1]


def f1_threshold(y_true, proba):
    grid = np.linspace(0.01, 0.99, 99)
    scores = [
        f1_score(y_true, proba >= threshold, zero_division=0)
        for threshold in grid
    ]
    best = int(np.argmax(scores))
    return float(grid[best]), float(scores[best])


def binary_scores(y_true, proba, threshold):
    y_true = np.asarray(y_true).astype(int)
    predicted = (proba >= threshold).astype(int)
    return {
        "rows": int(len(y_true)),
        "positives": int(y_true.sum()),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "auc": float(roc_auc_score(y_true, proba)),
    }


eval_proba = predict_proba(X_eval)
threshold, _ = f1_threshold(y_eval.to_numpy(), eval_proba)
print(f"F1 threshold from the eval holdout: {threshold:.2f}")

X_val, y_val = features_and_labels(val)
score_rows = [
    {"split": "train.csv", **binary_scores(y_train, predict_proba(X_train), threshold)},
    {"split": "val.csv", **binary_scores(y_val, predict_proba(X_val), threshold)},
    {"split": "train.csv fit rows", **binary_scores(y_fit, predict_proba(X_fit), threshold)},
    {"split": "train.csv eval holdout", **binary_scores(y_eval, eval_proba, threshold)},
]
score_table = pd.DataFrame(score_rows).set_index("split")
score_table

F1 threshold from the eval holdout: 0.89


,rows,positives,f1,auc
split,,,,
train.csv,40325,810,0.414894,0.953562
val.csv,34315,540,0.496028,0.974858
train.csv fit rows,32230,600,0.437393,0.975091
train.csv eval holdout,8065,210,0.365019,0.901302
